# Build an ITCH adapter

Define the protocol in Python, connect a source, and serve its books in the terminal. All of the protocol definition is below.

Start with the short recorded-input example, then connect the real source and open the terminal. Price and quantity outputs use integer units at the declared precisions.

In [1]:
from pathlib import Path
from time import perf_counter
import sys

root = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / "rust/crates/lobo_replay").is_dir())
sys.path.insert(0, str(root / "notebooks"))
from adapter_examples import wait_for_book
import pandas as pd
from IPython.display import display, HTML
from lobo.server import server_context

## Define the protocol

These declarations are the adapter definition. Edit the layouts, conditions or mappings here to change how your source drives the books.

In [2]:
from lobo.replay.adapters import CustomAdapter, Protocol
from lobo.replay.adapters import expressions as le
from lobo.replay.adapters import models as lm


def protocol() -> Protocol:
    register = lm.Register(
        symbol=le.Field("symbol"),
        key=le.Variable("key"),
        price_decimals=4,
        quantity_decimals=0,
    )
    add_fields = {
        "id": lm.UInt(11, 8),
        "side": lm.Text(19, 1),
        "quantity": lm.UInt(20, 4),
        "symbol": lm.Text(24, 8),
        "price": lm.UInt(32, 4),
    }
    add = lm.Add(
        id=le.Field("id"),
        side=le.Field("side").map({"B": "buy", "S": "sell"}),
        price=le.Field("price"),
        quantity=le.Field("quantity"),
    )
    reduction = {"id": lm.UInt(11, 8), "quantity": lm.UInt(19, 4)}
    return Protocol(
        lm.Binary(
            length=lm.UInt(0, 2),
            tag=lm.UInt(0, 1),
            key=lm.UInt(1, 2),
            timestamp=lm.UInt(5, 6),
            records={
                "S": lm.Record(
                    size=12,
                    fields={"event": lm.Text(11, 1)},
                    actions=(le.When(le.Field("event").eq("S"), lm.DirectoryComplete()),),
                ),
                "R": lm.Record(
                    size=39, fields={"symbol": lm.Text(11, 8)}, actions=(register,)
                ),
                "A": lm.Record(size=36, fields=add_fields, actions=(register, add)),
                "F": lm.Record(size=40, fields=add_fields, actions=(register, add)),
                "E": lm.Record(
                    size=31,
                    fields=reduction,
                    actions=(lm.Execute(id=le.Field("id"), quantity=le.Field("quantity")),),
                ),
                "C": lm.Record(
                    size=36,
                    fields={**reduction, "price": lm.UInt(32, 4)},
                    actions=(
                        lm.Execute(
                            id=le.Field("id"),
                            quantity=le.Field("quantity"),
                            price=le.Field("price"),
                        ),
                    ),
                ),
                "X": lm.Record(
                    size=23,
                    fields=reduction,
                    actions=(lm.Cancel(id=le.Field("id"), quantity=le.Field("quantity")),),
                ),
                "D": lm.Record(
                    size=19,
                    fields={"id": lm.UInt(11, 8)},
                    actions=(lm.Remove(id=le.Field("id")),),
                ),
                "U": lm.Record(
                    size=35,
                    fields={
                        "id": lm.UInt(11, 8),
                        "new_id": lm.UInt(19, 8),
                        "quantity": lm.UInt(27, 4),
                        "price": lm.UInt(31, 4),
                    },
                    actions=(
                        lm.Replace(
                            id=le.Field("id"),
                            new_id=le.Field("new_id"),
                            quantity=le.Field("quantity"),
                            price=le.Field("price"),
                        ),
                    ),
                ),
            },
        )
    )

## Construct, then consume input

`CustomAdapter(protocol(), source, ...)` validates and compiles the declaration with Cranelift during construction. There is no separate compile call. The source then supplies raw bytes to the compiled packet program. Changing a declaration creates a different program; identical definitions can reuse the process-local cache.

The timings below separate construction from `start()` → `wait()`. Construction includes declaration conversion and preparation; it is **not a compiler-only measurement**. Starting also includes worker setup and, when serving, preparation for publishing to observers. These tiny examples demonstrate behavior, not throughput.

Streaming binary fields load from the byte offsets declared above. This example uses `start()` so it exercises the compiled packet program. `run(concurrent=True)` uses the separate prepared binary path for grouped full-file replay.

In [3]:
# Fixed input: bid 100 @ 1.0000, ask 100 @ 1.0100, then execute 40 of the ask.
raw_records = bytes.fromhex(
    "00 24 41 00 01 00 00 00 00 00 00 00 64 00 00 00 00 00 00 00 01 42 00 00 00 64 41 41 50 4c 20 20 20 20 00 00 27 10"
    "00 24 41 00 01 00 00 00 00 00 00 00 64 00 00 00 00 00 00 00 02 53 00 00 00 64 41 41 50 4c 20 20 20 20 00 00 27 74"
    "00 1f 45 00 01 00 00 00 00 00 00 00 c8 00 00 00 00 00 00 00 02 00 00 00 28 00 00 00 00 00 00 00 01"
)
definition = protocol()
source = lm.Source.packets([raw_records])
started = perf_counter()
fixture_adapter = CustomAdapter(
    definition, source, name="ITCH recording", symbol="AAPL", scope=["AAPL"],
    mode="replay", level="l3", timezone="America/New_York",
)
construction_ms = (perf_counter() - started) * 1000

with fixture_adapter:
    started = perf_counter()
    fixture_adapter.start()
    fixture_adapter.wait()
    completion_ms = (perf_counter() - started) * 1000
    status = fixture_adapter.status()
    levels = pd.DataFrame(fixture_adapter.levels("AAPL"))
    queue = pd.DataFrame(fixture_adapter.queue("AAPL", "sell", 10100, 10100),
                         columns=["order_id", "price", "quantity", "timestamp_ns"])
    assert status["messages"] == 3
    assert queue["quantity"].tolist() == [60]
    display(levels, queue)
    fixture_adapter.simulate("buy", 25)
    display(pd.DataFrame(fixture_adapter.simulation_report()["executions"]))

print(f"Construction: {construction_ms:.3f} ms | Start to completion: {completion_ms:.3f} ms")
print(f"Consumed {status['bytes']:,} bytes / {status['messages']} records; ask remaining: 60")

,hidden,orders,price,quantity,side
0,0,1,10000,100,buy
1,0,1,10100,60,sell


,order_id,price,quantity,timestamp_ns
0,2,10100,60,100


,price,quantity,sequence,simulated,timestamp_ns
0,10100,25,1,True,200


Construction: 2.294 ms | Start to completion: 0.088 ms
Consumed 109 bytes / 3 records; ask remaining: 60


## Connect the source and serve its books

The same declaration drives the full local file. The scope fixes which books the session runs. This section requires the local Nasdaq file.

In [4]:
symbol = 'AAPL'
definition = protocol()
started = perf_counter()
adapter = CustomAdapter(
    definition, lm.Source.file(root / "data/NASDAQ/01302020.NASDAQ_ITCH50"),
    name='ITCH', symbol=symbol, scope=[symbol], mode='replay', level='l3',
    timezone='America/New_York',
)
print(f"Construction (may reuse compiled code): {(perf_counter() - started) * 1000:.3f} ms")

Construction (may reuse compiled code): 27.772 ms


Open the link to see the charts and use the simulation controls. The server stays running until the cleanup cell.

Run the cells individually to keep the terminal open while you explore. Run All reaches cleanup and closes it.

In [5]:
terminal = server_context(adapters=[adapter], port=0)
display(HTML(f'<a href="{terminal.url}" target="_blank">Open order-book terminal</a>'))

In [6]:
levels = wait_for_book(adapter, symbol)
display(pd.DataFrame(levels).head(12))
adapter.status()

,hidden,orders,price,quantity,side
0,0,1,3213100,100,buy
1,0,1,3212000,500,buy
2,0,1,3211500,325,buy
3,0,1,3211000,500,buy
4,0,1,3210100,30,buy
5,0,4,3210000,86,buy
6,0,3,3209000,2021,buy
7,0,1,3207800,100,buy
8,0,1,3202000,525,buy
9,0,2,3201700,130,buy


{'books': 1,
 'bytes': 53485562,
 'checksum_checks': 0,
 'checksum_failures': 0,
 'clock_ns': 17266533472599,
 'complete': False,
 'messages': 1834632,
 'symbol': 'AAPL',
 'synchronized': True}

Preview a market order and inspect its execution report.

In [7]:
adapter.simulate("buy", 100)
adapter.simulation_report()

{'alternate_timeline': False,
 'average_price': 3215500.0,
 'complete': True,
 'executions': [{'price': 3215500,
   'quantity': 41,
   'sequence': 1,
   'simulated': True,
   'timestamp_ns': 17326991435174},
  {'price': 3215500,
   'quantity': 59,
   'sequence': 2,
   'simulated': True,
   'timestamp_ns': 17326991435174}],
 'filled': 100,
 'ignored': 0,
 'order_id': 'f7d710a8-5ae3-45a7-80cd-15bb075da046',
 'remaining': 0,
 'requested': 100,
 'simulated': True,
 'stopped': True,
 'symbol': 'AAPL'}

Run this cell when finished exploring the terminal.

In [8]:
terminal.close()

## Recorded performance results

The September 9, 2026 paired benchmark for **4,096 streaming records** measured **600.2 µs** for the existing adapter and **825.9 µs** for the compiled custom adapter, a **1.38×** paired ratio. Streaming parity has not been reached. These are saved benchmark results, not timings from this notebook. The benchmark excludes construction and compilation; the demonstration above includes worker startup.

See [validation and reproduction commands](../python/examples/VALIDATION.md) for all four feeds and the timing boundaries.

The separate full-file all-ticker runs measured **50.5 s existing / 43.5 s custom**, both below 100 s. They use grouped binary replay, not the streaming path above, and are single samples.

Run from the repository root:

```sh
make bench-py PY_BENCH_SCENARIO=custom-all-tickers
make bench-py PY_BENCH_SCENARIO=custom-aapl-single-threaded
```

The same Python declaration can run directly into books with `CustomAdapter(protocol(), Source.file(path), ...).run(concurrent=True)`.